In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import scipy
import numpy as np
import pandas as pd
import sys
import multivelo as mv
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt
import dynamo as dyn
import anndata
import os.path
import pickle as pickle
import time
from os.path import exists
import unitvelo as utv
sys.path.append("/..")
scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
np.set_printoptions(suppress=True)
method = 'MultiVelo'

E0000 00:00:1734346905.292797  266221 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1734346905.347890  266221 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


(Running UniTVelo 0.2.5.2)
2024-12-16 11:01:51


In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

In [ ]:
df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

Empty DataFrame
Columns: [Mean, Time(s)]
Index: []
Empty DataFrame
Columns: [Mean, Time(s)]
Index: []


In [ ]:
for dataset in datasets:
    print(dataset)
    adata_rna = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
    start = time.time()
    scv.pp.filter_genes_dispersion(adata_rna, n_top_genes=2000,log=False)
    dyn.tl.neighbors(adata_rna,n_neighbors=30)
    scv.pp.moments(adata_rna)
    adata_result = mv.recover_dynamics_chrom(adata_rna=adata_rna, 
                                        max_iter=5, 
                                        init_mode="invert",
                                        parallel=True,
                                        n_jobs = 10,
                                        save_plot=False,
                                        rna_only=True,
                                        fit=True,
                                        n_anchors=500
                                        )
    mv.velocity_graph(adata_result)
    mv.latent_time(adata_result)
    end = time.time()
    fix, ax = plt.subplots(1, 1, figsize = (8, 6))
    mv.velocity_embedding_stream(adata_result, basis='umap', show=True,color='clusters',ax=ax)
    plt.savefig(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.svg')
    # Calculate performance metrics:
    file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
    ground_truth = pickle.load(file)
    metrics = utv.evaluate(adata_result, ground_truth, 'clusters', 'velo_s_norm')
    if exists(save_dir + method +'/'+ '_CBDir_scores.csv'):
        tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
    else:
        tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
        tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
    ##CBDC_scores
    cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
    tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
    # df_CB=df_CB.append(pd.DataFrame([[np.mean(cb_score), end-start]],columns=df_CB.columns,index=[dataset]))
    df_CB = pd.concat([df_CB, pd.DataFrame([[np.mean(cb_score), end - start]], columns=df_CB.columns, index=[dataset])])
    tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
    ##ICCoh_scores
    IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                for x in metrics['In-cluster Coherence'].keys()]
    tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]
    # df_IC=df_IC.append(pd.DataFrame([[np.mean(IC_score), end-start]],columns=df_IC.columns,index=[dataset]))
    df_IC = pd.concat([df_IC, pd.DataFrame([[np.mean(IC_score), end - start]], columns=df_IC.columns, index=[dataset])])
    tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
    adata=adata_result
    adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [ ]:
print(df_CB)
print(df_IC)
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores1.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores1.csv')